# Notebook 09: Putting It All Together

**Congratulations!** You have learned every piece of a production MLOps system.

Over the course of this workshop, you have built:

| Notebook | What You Built |
|----------|----------------|
| **00 -- Introduction** | Understanding of the MLOps landscape and lifecycle |
| **01 -- Data Engineering** | Synthetic data generation, validation, and quality gates |
| **02 -- Experiment Tracking** | MLflow for logging experiments, comparing runs, and model registry |
| **03 -- Model Development** | Baseline, XGBoost, and LSTM models with proper evaluation |
| **04 -- Training Pipelines** | Automated, reproducible, end-to-end training pipelines |
| **05 -- Model Serving** | FastAPI REST API for real-time and batch predictions |
| **06 -- Monitoring** | Data drift detection, model quality tracking, and alerting |
| **07 -- CI/CD & Deployment** | GitHub Actions, Docker, Kubernetes |
| **08 -- Orchestration** | Airflow DAGs for training, inference, and monitoring |

Now it is time to see **everything work together** in a single end-to-end demonstration.

### Architecture Overview

```
                        +------------------+
                        |   GitHub Actions  |
                        |   (CI/CD)         |
                        +--------+---------+
                                 |
                                 | deploys
                                 v
+------------+          +--------+---------+          +----------------+
|  Raw Data  |   --->   |  Airflow         |   --->   |  MLflow        |
|  (sensors, |          |  (orchestration)  |          |  (experiments, |
|   weather) |          |                   |          |   model reg.)  |
+------------+          |  Training DAG     |          +-------+--------+
                        |  Inference DAG    |                  |
                        |  Monitoring DAG   |                  | loads model
                        +--------+----------+                  |
                                 |                             v
                                 |                    +--------+---------+
                                 |                    |  FastAPI         |
                                 |                    |  (serving)       |
                                 |                    |  /predict        |
                                 |                    +--------+---------+
                                 |                             |
                                 v                             v
                        +--------+----------+         +--------+---------+
                        |  Prometheus +      |         |  Predictions     |
                        |  Grafana           |  <---   |  (stored,        |
                        |  (monitoring)      |         |   monitored)     |
                        +--------------------+         +------------------+
```

---
## 2. End-to-End Demo

Let's run the entire ML lifecycle in a single notebook -- from raw data to monitored predictions. Each step uses the actual production code from our `src/energy_forecast/` package.

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np

print("=" * 60)
print("STEP 1: Generate and Validate Data")
print("=" * 60)

from energy_forecast.data.synthetic import SyntheticDataGenerator
from energy_forecast.data.validator import DataValidator

gen = SyntheticDataGenerator(
    num_buildings=3,
    start_date="2023-01-01",
    end_date="2023-12-31",
    random_seed=42,
)
df = gen.generate()

validator = DataValidator()
result = validator.validate_raw_data(df)
print(f"Generated {len(df)} rows, Valid: {result.is_valid}")

In [ ]:
print("\nSTEP 2: Engineer Features")
print("=" * 60)

from energy_forecast.features.engineering import FeatureEngineer

engineer = FeatureEngineer()
df = engineer.create_time_features(df)
df = engineer.create_lag_features(df, "energy_demand_kwh", [1, 2, 3, 24])
df = engineer.create_rolling_features(df, "energy_demand_kwh", [6, 24])
df = engineer.create_weather_features(df)
df = engineer.create_calendar_features(df)
df = df.dropna()

feature_names = engineer.get_feature_names(df)
print(f"Engineered {len(feature_names)} features")
print(f"Dataset shape after feature engineering: {df.shape}")
print(f"Sample features: {feature_names[:10]}")

In [ ]:
print("\nSTEP 3: Train Model")
print("=" * 60)

from energy_forecast.models.xgboost_model import XGBoostForecaster
from sklearn.preprocessing import StandardScaler

X = df[feature_names].values
y = df["energy_demand_kwh"].values
n = len(X)

# Time-based split: 70% train, 15% validation, 15% test
X_train, y_train = X[:int(n * 0.7)], y[:int(n * 0.7)]
X_test, y_test = X[int(n * 0.85):], y[int(n * 0.85):]

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

model = XGBoostForecaster(n_estimators=300, max_depth=6, learning_rate=0.05)
model.fit(X_train_s, y_train)

print(f"Model trained on {len(X_train)} samples")
print(f"Test set: {len(X_test)} samples")
print("Model trained!")

In [ ]:
print("\nSTEP 4: Evaluate")
print("=" * 60)

from energy_forecast.evaluation.metrics import MetricsCalculator

preds = model.predict(X_test_s)
metrics = MetricsCalculator.compute_all(y_test, preds)

print("Evaluation Metrics:")
for k, v in metrics.items():
    print(f"  {k}: {v:.4f}")

In [ ]:
print("\nSTEP 5: Serve (API Test)")
print("=" * 60)

from fastapi.testclient import TestClient
from energy_forecast.serving.app import create_app

client = TestClient(create_app())
health = client.get("/health").json()
print(f"API Health: {health['status']}")
print(f"API is live and ready to serve predictions!")

In [ ]:
print("\nSTEP 6: Monitor for Drift")
print("=" * 60)

from energy_forecast.monitoring.drift import DriftDetector

detector = DriftDetector(reference_data=df.head(5000))
report = detector.detect_drift(df.tail(5000))
print(f"Drift detected: {report.is_drifted}")

print("\n" + "=" * 60)
print("ALL STEPS COMPLETE!")
print("=" * 60)
print("\nYou just ran the entire MLOps lifecycle:")
print("  1. Data generation and validation")
print("  2. Feature engineering")
print("  3. Model training")
print("  4. Model evaluation")
print("  5. API serving (health check)")
print("  6. Drift monitoring")
print("\nIn production, Airflow runs these steps automatically,")
print("Docker packages them, and Kubernetes scales them.")

---
## 3. What You Have Learned

Here is a checklist of every skill you have acquired in this workshop:

### Data Engineering
- [x] Generate realistic synthetic data for ML development
- [x] Validate data quality with automated checks
- [x] Build data pipelines that catch problems before they reach the model

### Feature Engineering
- [x] Create time-based features (hour, day, month, etc.)
- [x] Create lag and rolling window features for time series
- [x] Create weather and calendar features
- [x] Build reproducible feature engineering pipelines

### Experiment Tracking
- [x] Log experiments with MLflow (parameters, metrics, artifacts)
- [x] Compare experiments side-by-side
- [x] Use the MLflow Model Registry to version and stage models

### Model Development
- [x] Build baseline models for comparison
- [x] Train XGBoost gradient boosting models
- [x] Evaluate models with multiple metrics (MAE, RMSE, MAPE, R-squared)
- [x] Use time-based train/validation/test splits

### Training Pipelines
- [x] Build automated, end-to-end training pipelines
- [x] Make pipelines reproducible with configuration files
- [x] Add quality gates that prevent bad models from being registered

### Model Serving
- [x] Build a REST API with FastAPI
- [x] Implement health checks and input validation
- [x] Test APIs programmatically with TestClient

### Monitoring
- [x] Detect data drift using statistical tests
- [x] Track model performance over time
- [x] Set up alerts when thresholds are violated

### CI/CD and Deployment
- [x] Read and understand GitHub Actions CI pipelines
- [x] Write Dockerfiles for ML applications
- [x] Use Docker Compose to run multi-service systems
- [x] Understand Kubernetes deployments, services, and auto-scaling

### Orchestration
- [x] Understand Airflow DAGs, tasks, operators, and XCom
- [x] Read and understand training, inference, and monitoring DAGs
- [x] Understand how DAGs work together to form a self-maintaining system

---
## 4. Where to Go From Here

You now have a solid foundation in MLOps. Here are areas to explore next:

### Advanced Deployment Strategies
- **A/B Testing**: Serve two models simultaneously and compare their real-world performance. Route 10% of traffic to the new model and 90% to the current one.
- **Shadow Deployment**: Run the new model alongside the old one, but only serve the old model's predictions. Compare results without any risk.
- **Canary Releases**: Gradually shift traffic from the old model to the new one. If metrics degrade, automatically roll back.

### Advanced Monitoring
- **Prediction Explainability**: Use SHAP or LIME to explain why the model made each prediction. Monitor for changes in feature importance over time.
- **Label Delay Handling**: In many real-world systems, ground truth labels arrive days or weeks after predictions. Learn how to handle this delay.
- **Custom Drift Tests**: Go beyond simple distribution tests. Monitor for concept drift (the relationship between features and target changes, even if distributions stay the same).

### Cloud MLOps Platforms
- **AWS SageMaker**: End-to-end ML platform with built-in training, serving, and monitoring.
- **Google Vertex AI**: Managed ML platform with AutoML and custom model support.
- **Azure ML**: Microsoft's ML platform with strong integration into the Azure ecosystem.
- These platforms handle much of the infrastructure for you, but understanding the fundamentals (which you now have) is essential for using them effectively.

### Feature Stores
- **Feast**: An open-source feature store that ensures the same features are used in training and serving.
- **Tecton**: A managed feature platform for real-time ML.
- Feature stores solve the training-serving skew problem -- one of the most common causes of production ML failures.

### Model Optimization
- **ONNX**: Convert models to a universal format for faster inference.
- **Quantization**: Reduce model size and inference time by using lower precision.
- **Model Distillation**: Train a smaller model to mimic a larger one.

### The Best Way to Learn

The best way to deepen your MLOps skills is to **build something real**. Take a problem you care about, collect real data, and put a model into production. Every real deployment teaches lessons that no workshop can.

Start simple. Add complexity incrementally. Ship early, monitor everything, and iterate.

---
## 5. Thank You

Congratulations on completing the MLOps workshop! You have gone from zero to a production-grade ML system, understanding every layer along the way.

### Quick Reference: Make Commands

Here are the key commands for working with this project:

```bash
# Setup
make install-dev          # Install with all dev dependencies

# Code quality
make lint                 # Run Ruff linter
make format               # Auto-format code
make type-check           # Run mypy type checker

# Testing
make test                 # Run all tests
make test-unit            # Run unit tests only
make test-integration     # Run integration tests only
make test-cov             # Run tests with coverage

# Running the system
make docker-up            # Start all services
make docker-down          # Stop all services
make train                # Run training pipeline locally
make serve                # Start the prediction API locally
make seed                 # Generate synthetic data
make monitor              # Generate monitoring report
```

### Key URLs (when running with Docker Compose)

| Service | URL |
|---------|-----|
| FastAPI (predictions) | http://localhost:8000 |
| MLflow (experiments) | http://localhost:5000 |
| Airflow (orchestration) | http://localhost:8080 |
| Grafana (dashboards) | http://localhost:3000 |
| Prometheus (metrics) | http://localhost:9090 |
| MinIO (artifacts) | http://localhost:9001 |

### Further Reading

- [MLOps: Continuous delivery and automation pipelines in ML](https://cloud.google.com/architecture/mlops-continuous-delivery-and-automation-pipelines-in-machine-learning) -- Google's MLOps whitepaper
- [Made With ML](https://madewithml.com/) -- Comprehensive MLOps course by Goku Mohandas
- [Designing Machine Learning Systems](https://www.oreilly.com/library/view/designing-machine-learning/9781098107956/) -- Book by Chip Huyen
- [Evidently AI Blog](https://www.evidentlyai.com/blog) -- Deep dives on ML monitoring and drift
- [MLflow Documentation](https://mlflow.org/docs/latest/index.html)
- [Apache Airflow Documentation](https://airflow.apache.org/docs/)
- [FastAPI Documentation](https://fastapi.tiangolo.com/)

---

**You did it.** You understand the full MLOps lifecycle -- from raw data to production monitoring, from local development to Kubernetes at scale. Now go build something amazing.